# Tutorial: Flash-Radiomics scalar and spatial extraction

This tutorial is for users who installed Flash-Radiomics from PyPI or a wheel and want to extract scalar features and spatial maps from NIfTI or DICOM inputs. The example paths point to the IBSI-1 CT radiomics phantom when the notebook is run from its repository location; replace them with your own image and raster label map when using the installed package independently.

Prerequisites:

- Install runtime dependencies and then install `flash-radiomics` in the active Python environment.
- Update `IMAGE_PATH` and `MASK_PATH` for your own NIfTI files or DICOM series directory and raster label map.
- Select only a backend reported as available by `detect_backends()`.

By the end, you will be able to configure extraction through direct function arguments or a YAML file, override YAML settings at the call site, and inspect the combined HDF5 schema.


## Outline

1. Locate and inspect the bundled phantom.
2. Create a reusable YAML configuration.
3. Run extraction using direct function arguments.
4. Run extraction from YAML and override one YAML setting in the function call.
5. Inspect the fixed HDF5 hierarchy and one spatial map.
6. Inspect one spatial map.
7. Complete the YAML exercise.


In [1]:
from __future__ import annotations

from pathlib import Path

import h5py
import SimpleITK as sitk

from flash_radiomics import (
    RadiomicsFeatureExtractor,
    detect_backends,
    extract_scalar_and_spatial,
)


# Resolve inputs and outputs from the notebook working directory.
PROJECT_ROOT = Path.cwd().resolve()

# Example NIfTI input:
# IMAGE_PATH = Path("inputs/image.nii.gz").resolve()
# Example DICOM series input:
# IMAGE_PATH = Path("inputs/dicom_series").resolve()
IMAGE_PATH = Path("../reproducibility/data/ibsi/data_sets/ibsi_1_ct_radiomics_phantom/nifti/image/phantom.nii.gz").resolve()
MASK_PATH = Path("../reproducibility/data/ibsi/data_sets/ibsi_1_ct_radiomics_phantom/nifti/mask/mask.nii.gz").resolve()

OUTPUT_DIR = PROJECT_ROOT / "flashrad_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
YAML_CONFIG_PATH = OUTPUT_DIR / "extraction_override.yaml"

BACKEND = "cpu"
CASE_ID = "case_001"
ROI_ID = "label_1"

if not IMAGE_PATH.exists():
    raise FileNotFoundError(f"Image input does not exist: {IMAGE_PATH}")

if not MASK_PATH.is_file():
    raise FileNotFoundError(f"Mask input does not exist: {MASK_PATH}")

print("Available backends:", detect_backends())
print("Selected backend:", BACKEND)
print("Image:", IMAGE_PATH)
print("Mask:", MASK_PATH)
print("Output directory:", OUTPUT_DIR)

Available backends: {'cpu': True, 'mps': True, 'cuda': False}
Selected backend: cpu
Image: /Users/sding2/Desktop/00_PHD/00_PROJECT/MDA/ACTIVE/1_BRANCH/FLASH_PROD/flash-radiomics/reproducibility/data/ibsi/data_sets/ibsi_1_ct_radiomics_phantom/nifti/image/phantom.nii.gz
Mask: /Users/sding2/Desktop/00_PHD/00_PROJECT/MDA/ACTIVE/1_BRANCH/FLASH_PROD/flash-radiomics/reproducibility/data/ibsi/data_sets/ibsi_1_ct_radiomics_phantom/nifti/mask/mask.nii.gz
Output directory: /Users/sding2/Desktop/00_PHD/00_PROJECT/MDA/ACTIVE/1_BRANCH/FLASH_PROD/flash-radiomics/tutorials/flashrad_output


## 1. Inspect the bundled 3D phantom

Flash Radiomics accepts paths or `SimpleITK.Image` objects. Here we load the files only to inspect their geometry; extraction cells below pass the file paths directly.


In [2]:
image = sitk.ReadImage(str(IMAGE_PATH))
mask = sitk.ReadImage(str(MASK_PATH))
{
    "image_size_xyz": image.GetSize(),
    "spacing_xyz": image.GetSpacing(),
    "dimension": image.GetDimension(),
    "mask_labels": sorted(set(sitk.GetArrayViewFromImage(mask).ravel().tolist())),
}


{'image_size_xyz': (204, 201, 60),
 'spacing_xyz': (0.9769999980926514, 0.9769999980926514, 3.0),
 'dimension': 3,
 'mask_labels': [0, 1]}

## 2. Two configuration approaches

Flash-Radiomics accepts extraction settings in two ways. Approach 1 passes settings directly as function keyword arguments. Approach 2 passes a YAML file through `config`; keyword arguments supplied in the same call override matching settings loaded from YAML. The YAML feature maps independently select scalar and spatial outputs.


In [3]:
YAML_CONFIG_PATH.write_text(
    """setting:
  label: 1
  binWidth: 25.0
  distances: [1]
  normalize: false
  additionalInfo: true
  hdf5Compression: lzf
voxelSetting:
  kernelRadius: 1
  maskedKernel: true
  initValue: 0.0
  voxelBatch: -1
imageType:
  Original: {}
scalarFeatureClass:
  firstorder: [Mean, Variance]
  glcm: [Contrast]
spatialFeatureClass:
  firstorder: [Mean]
  glcm: [Contrast]
""",
    encoding="utf-8",
)


def make_extractor(
    mode: str,
    config_path: str | Path | None = None,
    backend: str = BACKEND,
) -> RadiomicsFeatureExtractor:
    if mode not in {"scalar", "spatial"}:
        raise ValueError("mode must be 'scalar' or 'spatial'")

    if config_path is not None:
        return RadiomicsFeatureExtractor(Path(config_path), backend=backend, num_threads=1)

    extractor = RadiomicsFeatureExtractor(
        backend=backend,
        num_threads=1,
        binWidth=1.0,
        distances=[1],
        additionalInfo=False,
    )
    if mode == "scalar":
        extractor.enableAllScalarFeatures()
    else:
        extractor.enableAllSpatialFeatures()
    return extractor


## 3. Approach 1: direct function call

Call `extract_scalar_and_spatial` directly and provide settings as keyword arguments. With no configuration file, the helper enables every supported scalar and spatial feature family. This approach is convenient for short scripts and interactive work.


In [4]:
direct_run = extract_scalar_and_spatial(
    IMAGE_PATH,
    MASK_PATH,
    label=1,
    backend=BACKEND,
    num_threads=1,
    binWidth=25.0,
    distances=[1],
    normalize=False,
    additionalInfo=True,
    output_hdf5_path=OUTPUT_DIR / "direct_function_call.flashrad.h5",
    case_id=CASE_ID,
    roi_id=ROI_ID,
)

{
    "scalar_feature_count": sum(name.startswith("original_") for name in direct_run["scalar"]),
    "spatial_map_count": sum(isinstance(value, sitk.Image) for value in direct_run["spatial"].values()),
    "combined_hdf5": str(direct_run["hdf5_path"].relative_to(PROJECT_ROOT)),
}


No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'geometryTolerance': None, 'correctMask': False, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'binWidth': 25.0, 'binCount': None, 'binMinimum': None, 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'resegmentMode': 'absolute', 'resegmentShape': False, 'sigma': [], 'start_level': 0, 'level': 1, 'wavelet': 'coif1', 'gradientUseSpacing': True, 'lbp2DRadius': 1.0, 'lbp2DSamples': 8, 'lbp2DMethod': 'uniform', 'lbp3DLevels': 2, 'lbp3DIcosphereRadius': 1.0, 'lbp3DIcosphereSubdivision': 1, 'voxelArrayShift': 0, 'symmetricalGLCM': True, 'weightingNorm': None, 'gldm_a': 0, 'segmentClassWorkers': 0, 'cudaGlszmTieBreakLowestLabel': True, 'cudaGlszmDeterministicReduction': False, 'cudaGlszmAllowEarlyExit': True, 'cudaGlszmCclSyncInterval': 4, 'cudaGlszmCc

{'scalar_feature_count': 173,
 'spatial_map_count': 101,
 'combined_hdf5': 'flashrad_output/direct_function_call.flashrad.h5'}

## 4. Approach 2: YAML configuration with a function override

Pass the YAML path through `config` for a reusable extraction definition. This YAML selects a small feature subset and sets `binWidth: 25.0`. The function call supplies `binWidth=20.0`, which takes precedence over the YAML value while leaving its feature selections unchanged.


In [5]:
yaml_run = extract_scalar_and_spatial(
    IMAGE_PATH,
    MASK_PATH,
    config=YAML_CONFIG_PATH,
    binWidth=20.0,  # Overrides setting.binWidth from the YAML file.
    output_hdf5_path=OUTPUT_DIR / "yaml_override.flashrad.h5",
    backend=BACKEND,
    num_threads=1,
    label=1,
    case_id=CASE_ID,
    roi_id=ROI_ID,
)
{
    "scalar_outputs": sorted(name for name in yaml_run["scalar"] if name.startswith("original_")),
    "spatial_map_count": sum(isinstance(value, sitk.Image) for value in yaml_run["spatial"].values()),
    "effective_bin_width": 20.0,
    "combined_hdf5": str(yaml_run["hdf5_path"].relative_to(PROJECT_ROOT)),
}


Loading parameter file /Users/sding2/Desktop/00_PHD/00_PROJECT/MDA/ACTIVE/1_BRANCH/FLASH_PROD/flash-radiomics/tutorials/flashrad_output/extraction_override.yaml
Applying custom setting overrides: {'binWidth': 20.0}
Loading parameter file /Users/sding2/Desktop/00_PHD/00_PROJECT/MDA/ACTIVE/1_BRANCH/FLASH_PROD/flash-radiomics/tutorials/flashrad_output/extraction_override.yaml
Applying custom setting overrides: {'binWidth': 20.0}
Calculating features with label: 1
Loading image and mask
Adding image type "Original" with custom settings: {}
Calculating features for original image
Computing firstorder
Computing glcm
Starting spatial extraction
Calculating features with label: 1
Loading image and mask
Adding image type "Original" with custom settings: {}
Calculating features for original image


{'scalar_outputs': ['original_firstorder_Mean',
  'original_firstorder_Variance',
  'original_glcm_Contrast'],
 'spatial_map_count': 2,
 'effective_bin_width': 20.0,
 'combined_hdf5': 'flashrad_output/yaml_override.flashrad.h5'}

## 5. Inspect the saved HDF5 schema

Direct and YAML-configured extractions use the same paths. Only the lengths of `scalar_features/names`, `scalar_features/values`, `feature_names`, and the channel axis of `feature_maps` change. The root mode is `combined` because this file contains both payloads.


In [6]:
with h5py.File(yaml_run["hdf5_path"], "r") as handle:
    roi = handle[f"rois/{ROI_ID}"]
    hdf5_summary = {
        "schema_name": handle.attrs["schema_name"],
        "mode": handle.attrs["mode"],
        "roi_keys": sorted(roi.keys()),
        "scalar_feature_count": int(roi["scalar_features/names"].shape[0]),
        "spatial_map_shape_czyx": tuple(roi["feature_maps"].shape),
        "image_shape_czyx": tuple(roi["image"].shape),
        "mask_shape_czyx": tuple(roi["mask"].shape),
    }
hdf5_summary


{'schema_name': 'flashrad_hdf5',
 'mode': 'combined',
 'roi_keys': ['feature_maps',
  'feature_names',
  'image',
  'mask',
  'metadata',
  'scalar_features'],
 'scalar_feature_count': 3,
 'spatial_map_shape_czyx': (2, 28, 101, 102),
 'image_shape_czyx': (1, 28, 101, 102),
 'mask_shape_czyx': (1, 28, 101, 102)}

## 6. Inspect one spatial map

Spatial results are `SimpleITK.Image` objects with geometry preserved. Convert a map to NumPy only when array operations or visualization are needed.


In [7]:
map_name, feature_map = next(
    (name, value)
    for name, value in direct_run["spatial"].items()
    if isinstance(value, sitk.Image)
)
map_array = sitk.GetArrayFromImage(feature_map)
{
    "name": map_name,
    "array_shape_zyx": map_array.shape,
    "spacing_xyz": feature_map.GetSpacing(),
}


{'name': 'original_firstorder_Mean',
 'array_shape_zyx': (28, 101, 102),
 'spacing_xyz': (0.9769999980926514, 0.9769999980926514, 3.0)}

## Exercise

Copy the generated `extraction_override.yaml`, change the first-order selection to `[Mean, Median, Maximum]`, remove the GLCM family, and set `spatialFeatureClass: null`, and pass the copied path to `RadiomicsFeatureExtractor`. Set `exercise_config` below to that copied path, then compare the returned names with the answer shown when it is unset.


In [8]:
exercise_config = None

if exercise_config is not None:
    exercise_extractor = RadiomicsFeatureExtractor(
        exercise_config, backend=BACKEND, num_threads=1
    )
    exercise_result = exercise_extractor.execute(
        str(IMAGE_PATH),
        str(MASK_PATH),
        label=1,
        spatialMode=False,
        output_hdf5_path=OUTPUT_DIR / "exercise_scalar.flashrad.h5",
        case_id=CASE_ID,
        roi_id=ROI_ID,
    )
    print(sorted(name for name in exercise_result if name.startswith("original_")))
else:
    print("Expected names: original_firstorder_Mean, original_firstorder_Median, original_firstorder_Maximum")


Expected names: original_firstorder_Mean, original_firstorder_Median, original_firstorder_Maximum


## Notes

- Scalar and spatial writes to the same explicit HDF5 path are merged for the same case and ROI. Reusing a path intentionally updates that container.
- A custom YAML replaces the default feature selection. Function keyword arguments override matching global settings loaded from that YAML.
- Confirm that `detect_backends()["cuda"]` is true before requesting CUDA.
- Wheel installation needs no CUDA toolkit. Install Python dependencies separately.
- For larger studies, reuse an extractor with consistent settings and use case/ROI identifiers to keep outputs traceable.
